# S1 v3 pointer A: masked option-to-option attention + shared option start position

1. Numerically verify permutation invariance on real `Qwen/Qwen3-0.6B-Base` (expect 0 flips).
2. Train `--s1 --isolate-options --save-ckpt` (1000 steps, 4000 rows, batch 4, seq 832; single forward pass ~45-50 min).
3. Evaluate saved `ckpt_A` on full held-out sets across 2x T4 (`--flip-orders 5`, 95% CIs, collision & near-miss slices).

In [ ]:
import glob, os, shutil, subprocess, sys, time, pathlib, json
REPO = "/kaggle/working/Soro-Boro"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "upp-p0", "https://github.com/nisaral/Soro-Boro.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only", "origin", "upp-p0"], check=False)
os.chdir(REPO)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout, flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "bitsandbytes", "huggingface_hub"], check=True)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=True)

In [ ]:
import random, torch
sys.path.insert(0, os.path.join(REPO, "veyra", "python", "src"))
from soro.routing.upp import UPPConfig, UPPModel

check_m = UPPModel(
    UPPConfig(layout="query_first", head="dot", option_index=False, option_isolation=True),
    model_name="Qwen/Qwen3-0.6B-Base",
    torch_dtype=torch.float32,
).eval().cuda()
check_m.backbone.float()
crit = {
    "send_email": "send an email message to a recipient with subject and body",
    "send_sms": "send a text SMS message to a phone number",
    "get_weather": "check current weather forecast for a city",
    "search_docs": "search workspace documents matching a query",
    "__abstain__": "None of the listed tools or skills fit the user request.",
}
with torch.no_grad():
    p0_32, _ = check_m.score("User Request: text Alex that I am running 10 mins late", crit)
    keys = list(crit)
    max_diff_32 = 0.0
    tops_32 = {max(p0_32, key=p0_32.get)}
    rng = random.Random(7)
    for _ in range(5):
        perm = list(keys)
        rng.shuffle(perm)
        pk, _ = check_m.score("User Request: text Alex that I am running 10 mins late", {k: crit[k] for k in perm})
        max_diff_32 = max(max_diff_32, max(abs(p0_32[k] - pk[k]) for k in keys))
        tops_32.add(max(pk, key=pk.get))
print(f"Real Qwen3-0.6B-Base fp32 5-order check: max_prob_diff={max_diff_32:.6e}, distinct_tops={len(tops_32)}", flush=True)
assert max_diff_32 < 1e-4 and len(tops_32) == 1, f"FP32 invariance check failed: diff={max_diff_32}, tops={tops_32}"
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    p0_16, _ = check_m.score("User Request: text Alex that I am running 10 mins late", crit)
    max_diff_16 = 0.0
    tops_16 = {max(p0_16, key=p0_16.get)}
    rng = random.Random(7)
    for _ in range(5):
        perm = list(keys)
        rng.shuffle(perm)
        pk, _ = check_m.score("User Request: text Alex that I am running 10 mins late", {k: crit[k] for k in perm})
        max_diff_16 = max(max_diff_16, max(abs(p0_16[k] - pk[k]) for k in keys))
        tops_16.add(max(pk, key=pk.get))
print(f"Real Qwen3-0.6B-Base fp16 5-order check: max_prob_diff={max_diff_16:.6e}, distinct_tops={len(tops_16)}", flush=True)
assert max_diff_16 < 5e-3 and len(tops_16) == 1, f"FP16 invariance check failed: diff={max_diff_16}, tops={tops_16}"
del check_m
torch.cuda.empty_cache()

In [ ]:
t0 = time.time()
try:
    cmd = [
        sys.executable, "veyra/training/p0_upp_ablation.py",
        "--s1", "--isolate-options", "--configs", "A", "--save-ckpt",
        "--base-model", "Qwen/Qwen3-0.6B-Base",
        "--train-rows", "4000", "--eval-rows", "120", "--batch", "4",
        "--max-seq-len", "832", "--max-steps", "1000", "--log-every", "50",
        "--out", "runs/s1_v3",
    ]
    print(" ".join(cmd), flush=True)
    rc = subprocess.run(cmd).returncode
    print(f"train done in {(time.time()-t0)/60:.1f} min rc={rc}", flush=True)
    if rc != 0:
        raise SystemExit(rc)

    pack_dirs = [pathlib.Path(p).parent for p in glob.glob("/kaggle/input/**/eval_zeroshot.jsonl", recursive=True)]
    assert pack_dirs, "soro-s1-eval-pack dataset not mounted under /kaggle/input"
    pack = pack_dirs[0]
    print("eval pack:", pack, flush=True)
    eval_files = ",".join([
        f"selection={pack / 'selection_1330.jsonl'}",
        f"zeroshot={pack / 'eval_zeroshot.jsonl'}",
        f"external={pack / 'eval_external.jsonl'}",
        f"collisions={pack / 'eval_collisions_disjoint.jsonl'}",
        f"near_miss={pack / 'eval_near_miss_disjoint.jsonl'}",
    ])
    n_gpus = torch.cuda.device_count()
    t_ev = time.time()
    if n_gpus >= 2:
        procs = []
        for gi in range(2):
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gi))
            procs.append(subprocess.Popen([
                sys.executable, "veyra/training/eval_upp_full.py",
                "--ckpt", "runs/s1_v3/ckpt_A", "--eval-files", eval_files,
                "--flip-orders", "5", "--batch-size", "8",
                "--shard", f"{gi}/2", "--out", "runs/s1_v3/full_eval",
            ], env=env))
        rcs = [p.wait() for p in procs]
        if any(rcs):
            raise SystemExit(1)
        rc = subprocess.run([
            sys.executable, "veyra/training/eval_upp_full.py",
            "--merge-shards", "runs/s1_v3/full_eval/scored_shard_0.json,runs/s1_v3/full_eval/scored_shard_1.json",
            "--out", "runs/s1_v3/full_eval",
        ]).returncode
    else:
        rc = subprocess.run([
            sys.executable, "veyra/training/eval_upp_full.py",
            "--ckpt", "runs/s1_v3/ckpt_A", "--eval-files", eval_files,
            "--flip-orders", "5", "--batch-size", "8",
            "--out", "runs/s1_v3/full_eval",
        ]).returncode
    print(f"full eval done in {(time.time()-t_ev)/60:.1f} min rc={rc}", flush=True)
    if rc != 0:
        raise SystemExit(rc)

    ckpt = pathlib.Path("runs/s1_v3/ckpt_A/head.pt")
    if ckpt.exists():
        hs = torch.load(ckpt, map_location="cpu", weights_only=False)["head"]
        if "members.0.log_temp" in hs:
            print("final temp:", round(float(torch.exp(hs["members.0.log_temp"])), 5), flush=True)
    for fn, dst in [("runs/s1_v3/full_eval/full_eval.json", "/kaggle/working/v3_full_eval.json"),
                    ("runs/s1_v3/full_eval/full_eval.md", "/kaggle/working/v3_full_eval.md")]:
        if pathlib.Path(fn).exists():
            shutil.copy2(fn, dst)
    shutil.make_archive("/kaggle/working/s1_v3_results", "zip", "runs/s1_v3")
    print("wrote /kaggle/working/s1_v3_results.zip", flush=True)
finally:
    os.chdir("/kaggle/working")
    shutil.rmtree(REPO, ignore_errors=True)